# 13 — Multi-modal RAG: Giving StoryVerse AI Eyes

*Advanced series, part 3 of 3.*

Every document StoryVerse AI has read so far has been the same shape: clean, plain-text `.txt` files, one story per file. Real archives are never that polite. A real StoryVerse would have movie posters (images), old scanned scripts (PDFs full of pictures of text, not real text), and actual PDFs with selectable text. None of that is a `.txt` file, and none of it fits directly into `embedder.encode(...)`.

The idea that makes all of it work is simpler than it sounds: **turn everything into text first, then it's just RAG again.**


In [ ]:
import os
import io
import base64
import json
import faiss
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"
VISION_MODEL = "qwen/qwen3.6-27b"  # Groq's current image-input model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = os.path.join("..", "data")
STORIES_DIR = os.path.join(DATA_DIR, "stories")


## Modality 1 — images, turned into captions

A vision-capable model can describe an image in plain English. Once you have that description, it's a sentence like any other -- embed it, chunk it, retrieve it, exactly like a `.txt` file. Let's generate a simple StoryVerse movie poster and see what the model makes of it.


In [ ]:
from PIL import Image, ImageDraw
import random

def make_interstellar_poster() -> Image.Image:
    img = Image.new("RGB", (400, 600), color=(10, 10, 30))
    d = ImageDraw.Draw(img)
    d.ellipse([120, 180, 280, 340], outline=(255, 200, 50), width=6)  # wormhole ring
    d.ellipse([150, 210, 250, 310], fill=(0, 0, 0))  # black hole core
    random.seed(3)
    for _ in range(40):
        x, y = random.randint(0, 400), random.randint(0, 180)
        d.point((x, y), fill=(255, 255, 255))  # stars
    d.text((90, 500), "INTERSTELLAR", fill=(255, 255, 255))
    d.text((60, 530), "a StoryVerse original", fill=(180, 180, 200))
    return img


poster = make_interstellar_poster()
poster.save(os.path.join(DATA_DIR, "interstellar_poster.png"))
poster


In [ ]:
def image_to_data_url(image: Image.Image) -> str:
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f"data:image/png;base64,{b64}"


def caption_image(image: Image.Image, prompt: str = "Describe this image in 2-3 sentences, including any visible text.") -> str:
    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_to_data_url(image)}},
            ],
        }],
        max_completion_tokens=200,
        reasoning_effort="none",  # this model narrates its thinking inline unless told not to
    )
    return response.choices[0].message.content


poster_caption = caption_image(poster)
print(poster_caption)


That caption is now a plain sentence describing a wormhole, stars, and the word "INTERSTELLAR" -- a search for "black hole ring on a movie poster" can match it, even though nothing about `embedder.encode` knows what a pixel is. The image never touches the vector store; its *caption* does. This is the entire trick behind "multi-modal RAG" as most systems actually build it: convert everything to text, then reuse every pipeline from notebooks 04-08 unchanged.


## Modality 2 — scanned pages, turned into text via OCR

A caption describes a whole image at a glance. A scanned page is different: it's an image *of text*, and you want the text back out, not a summary of it. That's **OCR** (Optical Character Recognition). Let's simulate a scanned page -- a story rendered as a picture instead of real, selectable text -- and read it back.


In [ ]:
def make_scanned_page(text: str) -> Image.Image:
    import textwrap
    img = Image.new("RGB", (600, 300), color="white")
    d = ImageDraw.Draw(img)
    wrapped = textwrap.fill(text, width=45)
    d.multiline_text((30, 30), wrapped, fill="black", spacing=10)
    return img


dark_knight_text = open(os.path.join(STORIES_DIR, "dark_knight.txt")).read().strip().split("\n")[0]
scanned_page = make_scanned_page(dark_knight_text)
scanned_page.save(os.path.join(DATA_DIR, "scanned_page.png"))
scanned_page


In [ ]:
from rapidocr import RapidOCR

ocr_engine = RapidOCR()
ocr_result = ocr_engine(os.path.join(DATA_DIR, "scanned_page.png"))
ocr_text = " ".join(ocr_result.txts)

print("Original text: ", dark_knight_text)
print("OCR'd text:    ", ocr_text)


Compare the two lines above closely. OCR is not a perfect transcription -- expect a dropped letter, a merged word, a missing space here and there. This is exactly the **noisy documents** problem from notebook 09, just arriving from a scanner instead of a bad HTML export. The fix is the same one you already have: run `clean_document`-style normalization, then chunk and embed as usual. OCR text is *usable* text, not *clean* text -- treat it accordingly.


## Modality 3 — real PDFs

Not every PDF is a scanned image -- most modern ones have real, selectable text underneath. Let's build one (so this notebook doesn't depend on a file you don't have), then extract it properly.


In [ ]:
import fitz  # PyMuPDF

baahubali_text = open(os.path.join(STORIES_DIR, "baahubali.txt")).read().strip()

pdf_doc = fitz.open()
page = pdf_doc.new_page()
page.insert_text((50, 72), "Baahubali", fontsize=18)
page.insert_textbox((50, 100, 550, 750), baahubali_text, fontsize=11)
pdf_path = os.path.join(DATA_DIR, "baahubali.pdf")
pdf_doc.save(pdf_path)
pdf_doc.close()

print(f"Wrote {pdf_path}")


In [ ]:
import pymupdf4llm

pdf_text = pymupdf4llm.to_markdown(pdf_path)
print(pdf_text[:300], "...")


`pymupdf4llm` pulls the real text layer straight out of the PDF -- no OCR needed, because the text was never a picture to begin with. From here it's identical to notebook 06: chunk it, embed it, index it.


## One index, every modality

Time to put all three into the same vector store as the original 5 stories -- a poster caption, OCR'd scan text, and PDF-extracted text, sitting right alongside plain `.txt` files. Retrieval doesn't know or care which modality a chunk originally came from.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_documents(folder_path: str) -> list[dict]:
    docs = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename)) as f:
                docs.append({"filename": filename, "content": f.read()})
    return docs


splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
multimodal_store = []  # {"content", "metadata": {"source", "modality"}}

# Modality: plain text (notebooks 02-07)
for doc in load_documents(STORIES_DIR):
    for i, chunk in enumerate(splitter.split_text(doc["content"])):
        multimodal_store.append({"content": chunk, "metadata": {"source": doc["filename"], "modality": "text", "chunk_index": i}})

# Modality: image caption
multimodal_store.append({"content": poster_caption, "metadata": {"source": "interstellar_poster.png", "modality": "image_caption", "chunk_index": 0}})

# Modality: OCR'd scan
multimodal_store.append({"content": ocr_text, "metadata": {"source": "scanned_page.png", "modality": "ocr", "chunk_index": 0}})

# Modality: PDF text
for i, chunk in enumerate(splitter.split_text(pdf_text)):
    multimodal_store.append({"content": chunk, "metadata": {"source": "baahubali.pdf", "modality": "pdf", "chunk_index": i}})

print(f"Total chunks: {len(multimodal_store)}")
for modality in ("text", "image_caption", "ocr", "pdf"):
    count = sum(1 for d in multimodal_store if d["metadata"]["modality"] == modality)
    print(f"  {modality:15} {count} chunks")


In [ ]:
chunk_embeddings = embedder.encode([d["content"] for d in multimodal_store]).astype(np.float32)
faiss.normalize_L2(chunk_embeddings)

multimodal_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
multimodal_index.add(chunk_embeddings)


def retrieve_multimodal(query: str, top_k: int = 3, min_score: float = 0.3) -> list[dict]:
    query_vec = embedder.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = multimodal_index.search(query_vec, top_k)
    results = [{**multimodal_store[i], "score": float(scores[0][j])} for j, i in enumerate(indices[0])]
    return [r for r in results if r["score"] >= min_score]


test_queries = [
    "What does the Interstellar poster look like?",
    "Who does Shivudu fall in love with?",
    "What happens to Batman and Harvey Dent?",
]

for q in test_queries:
    top = retrieve_multimodal(q, top_k=1)
    if top:
        print(f"Q: {q}")
        print(f"   -> [{top[0]['metadata']['modality']}] {top[0]['metadata']['source']} (score {top[0]['score']:.2f})")
        print(f"      {top[0]['content'][:100]}...")
    else:
        print(f"Q: {q} -> nothing above threshold")
    print()


Look at the `modality` field in each result. The poster question should pull from `image_caption`, the Baahubali question from plain `text`, and the Batman question -- since that story only exists as a `.txt` file in our corpus -- also from `text`, never from the PDF (which only holds Baahubali) or the OCR scan (which only holds a Dark Knight snippet). Retrieval is doing exactly what it always does: ranking by meaning. It just happens that "meaning" now lives in chunks that arrived from three completely different pipelines.


## What we deliberately left out: audio and video

The full "multi-modal" idea usually includes audio and video too -- a movie's actual soundtrack, a trailer, a voiceover. The pattern doesn't change: run a speech-to-text model (Whisper is the standard choice) over the audio track, and the transcript becomes a text chunk exactly like everything above. This project doesn't have an audio transcription library installed, so rather than bolt one on half-built, this notebook stops at "here's the seam where it would plug in." If you need it: transcribe first, then everything from notebook 06 onward applies unchanged.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Vision-capable model | A model that accepts images as input alongside text |
| Image captioning | Asking a model to describe an image in words |
| OCR (Optical Character Recognition) | Extracting text from an image of text, not a summary of it |
| Modality | The original medium a piece of content came from (text, image, scan, PDF, audio) |

The whole notebook is one idea applied three times: **whatever the source, get it into plain text, then it's the same RAG pipeline from notebooks 04-08.** Captions, OCR output, and PDF-extracted text all end up as ordinary strings sitting in the same vector store as a `.txt` file, with a `modality` tag in the metadata as the only trace of where they came from.

**This closes the advanced series.** Notebook 11 added memory, notebook 12 added judgment about *when* to retrieve and what else to do instead, and this notebook added *what* can be retrieved from in the first place. Put together with notebooks 00-10, that's a genuinely production-shaped understanding of RAG -- not just calling a library, but knowing what every piece is doing underneath and why it's there.

**Exercises:**

- Generate a poster for a second story (Baahubali or Dark Knight) and confirm it's retrievable by a question about its visuals.
- Deliberately blur or shrink `scanned_page.png` before running OCR -- how much does text quality degrade, and does `clean_document` from notebook 09 need new rules to handle it?
- Add metadata filtering (notebook 09) so a query can be scoped to `modality: "text"` only, skipping images and scans entirely.
